# Jaccard Similarity 

In this section you will construct a similarity metric based on the Jaccard similarity coefficient. 

The Jaccard coefficient is based on sets operations, namely intersection and union.

![illustration of Jaccard similartiy](https://upload.wikimedia.org/wikipedia/commons/c/c7/Intersection_over_Union_-_visual_equation.png)

## 1. Load the dataset

Load the subset data you created in the previous notebook.

In [2]:
import pandas as pd
df_books_ratings = pd.read_csv('data/BX-Book-Ratings-Subset.csv', sep=';')

## 2. Make a set for each book entry

Now we need to construct a set for each book in our dataset. This set will be composed of User-IDs who rated the book. 

From our data frame, construct a python dictionary containing ISBNs as keys and a list (or a set) of User-IDs as values.

In [6]:
grouped_df = df_books_ratings.groupby(['ISBN'])['User-ID'].aggregate(list)
dict_isbn_groups = grouped_df.to_dict()

## 3. Jaccard distance function

Complete the `jaccard_distance` function below. It calculates the distance between two books based on who rated them. As input, it takes the user IDs for each book. It returns a similarity score: higher scores mean the lists are more similar.


In [7]:
def jaccard_distance(user_ids_isbn_a, user_ids_isbn_b):
                
    set_isbn_a = set(user_ids_isbn_a)
    set_isbn_b = set(user_ids_isbn_b)
    
    intersection = set_isbn_a.intersection(set_isbn_b)
    union = set_isbn_a.union(set_isbn_b)
        
    return len(intersection) / float(len(union))

## 4. Calculate distances 

Here is the ISBN of a book in our dataset (you can of course choose another one!). 

Can you calculate this book's jaccard distance from all the other books in the dataset?

In [8]:
a_book_isbn = '002542730X'

for isbn, users in dict_isbn_groups.items():
    if isbn != a_book_isbn:
        d = jaccard_distance(dict_isbn_groups[a_book_isbn], users)
        if d > 0.0:
            print(a_book_isbn + ' - ' + isbn + ' : d=' + str(d))

002542730X - 000649840X : d=0.031413612565445025
002542730X - 0006547834 : d=0.012903225806451613
002542730X - 0006550681 : d=0.012658227848101266
002542730X - 0006550789 : d=0.012738853503184714
002542730X - 0007110928 : d=0.006289308176100629
002542730X - 0007154615 : d=0.018072289156626505
002542730X - 0020198906 : d=0.024242424242424242
002542730X - 0020199600 : d=0.025157232704402517
002542730X - 002026478X : d=0.032432432432432434
002542730X - 0020427859 : d=0.025974025974025976
002542730X - 0020442009 : d=0.024691358024691357
002542730X - 0020442106 : d=0.019736842105263157
002542730X - 0020442203 : d=0.03954802259887006
002542730X - 0020442300 : d=0.012903225806451613
002542730X - 0020442408 : d=0.029411764705882353
002542730X - 0020442505 : d=0.031055900621118012
002542730X - 0020442602 : d=0.024691358024691357
002542730X - 0020446500 : d=0.019230769230769232
002542730X - 0020519001 : d=0.0189873417721519
002542730X - 0020532105 : d=0.00641025641025641
002542730X - 0020868308 

## 5. Function calculating distances 

Considering the code above, can you make a function that will take as input a given book's ISBN and calculate its distance from all other books in our dataset? 

In [9]:
# the following is based on the 'dict_isbn_groups' dict

def calculate_jaccard_distances(book_isbn):
    # first look if the book is in our data
    if book_isbn in dict_isbn_groups:
        # fetch the list of users who reviewed this book
        book_isbn_users = dict_isbn_groups[book_isbn]
        # create a working dict where we record the distances
        dict_distances = {}
        # iterate through other books
        for isbn, users in dict_isbn_groups.items():
            if isbn != book_isbn:
                # for each book, calculate the distance between this book's users and our book's users
                d = jaccard_distance(book_isbn_users, users)
                # record the distance in our working dict
                dict_distances[isbn] = d
        return dict_distances
    else:
        return None

# try it out with 'a_book_isbn'
calculate_jaccard_distances(a_book_isbn)

{'000649840X': 0.031413612565445025,
 '0006547834': 0.012903225806451613,
 '0006550681': 0.012658227848101266,
 '0006550789': 0.012738853503184714,
 '0007110928': 0.006289308176100629,
 '0007154615': 0.018072289156626505,
 '0020198906': 0.024242424242424242,
 '0020199600': 0.025157232704402517,
 '002026478X': 0.032432432432432434,
 '0020418809': 0.0,
 '0020427859': 0.025974025974025976,
 '0020442009': 0.024691358024691357,
 '0020442106': 0.019736842105263157,
 '0020442203': 0.03954802259887006,
 '0020442300': 0.012903225806451613,
 '0020442408': 0.029411764705882353,
 '0020442505': 0.031055900621118012,
 '0020442602': 0.024691358024691357,
 '0020446500': 0.019230769230769232,
 '0020519001': 0.0189873417721519,
 '0020532105': 0.00641025641025641,
 '0020868308': 0.024844720496894408,
 '0028604199': 0.12,
 '0028604202': 0.07051282051282051,
 '0060001453': 0.0,
 '0060002050': 0.006369426751592357,
 '0060002069': 0.0,
 '0060002492': 0.025477707006369428,
 '006000438X': 0.022598870056497175,

## 6. Function generating recommendations

Use your function to create a function that will give recommendations for a book. It should take an ISBN as input and return the ISBNs of the 5 most similar books, based on your jaccard similarity function.

In [13]:
def similar_books(isbn):
    distances = calculate_jaccard_distances(isbn)
    sorted_distances = sorted(
        distances.items(),
        reverse=True,
        key=lambda item: item[1]
    )
    top_5 = sorted_distances[:5]
    return [isbn for isbn, distance in top_5]

similar_books(a_book_isbn)

['0028604199', '080410526X', '0399501487', '0446606812', '0449907481']